# Baseball Attendance

Like many people interested in statistics and data, my favorite sport is baseball. Baseball yields an incredible amount of data, but for this demonstration I'd like to focus on something very approchable in baseball: attendance data. Starting in 2000, following the implementation of a revenue sharing system, Major League Baseball (MLB) began keeping detailed statistics of the number of tickets distributed for each game played. This isn't a true representation of attendance of course, since some fans who received tickets may not show up, however it gives a great general insight into if fans are interested in attending games for a particular team. 

The data provided by MLB is only numeric, and by default can only provide high level summary data. For example, we can evaluate the 2025 Detroit Tigers data and get some decent summary stats:

| stat | value |
| ---- | ----- |
| mean | 29,796 |
| std dev | 7,587 |
| min | 14,132 |
| median | 30,051 |
| max | 44,735 |

but these stats don't really tell us the full story. Sports media often likes to drive the idea of "storylines" throughout a season, and I think attendance tells its own story each season too. Just looking at the average, minimum, or maximum attendance for a team in a season doesn't really tell us anything about why people were attending games; combining this data with other periphreal information and plotting it on different types of charts can give us new insight into what happened during a season.

Particularly, I think there are a few things that can be very interesting when looking at the attendance data for MLB games:
- Which opponents drew the largest crowds for the team? Was it divisional rivals or was it the big market out of state teams?
- Are fans attending a game for a particular team more on certain days, like weekends? Are they showing up more for day or night games?
- If a team is playing important games (that may impact their playoff chances), will they draw a higher attendance?

## The Data

Baseball attendance data is available in a couple different places. [Retorsheet](https://retrosheet.org/) is great for almost any stats we want, including attendance data. However, it misses a few of extra points I want to explore, like the number of game number or the number of games back in the standings a team is at any given point. These numbers could be calculated ourselves, but for simplicity I'm going to download the data from [Baseball Reference](https://www.baseball-reference.com/) instead.

[Here's a sample](https://www.baseball-reference.com/teams/WSN/2019-schedule-scores.shtml) of what a team's schedule looks like on Baseball Reference. There are several columns that will give more insight into attendance trends:

- `Date`
- `Tm`: The team the data was scraped from
- `Unnamed: 4`: `NaN` or `@` to denote the home team
- `Opp`: The opposing team
- `Rank`: Where the team is in the standings
- `GB`: How many games behind first place the team is in the standings
- `D/N`: Denote a day or night game
- `Attendance`: The number of tickets officially distributed (NOTE: this is *not* how many fans actually attending the game)
- `cLI`: Championship Leverage Index, metric for how important the game is in order to win the championship ([further reading](https://www.sports-reference.com/blog/2020/09/__trashed-2/))
- `Orig. Scheduled`: When the game was originally scheduled. Also provides other information if a game was played at a different location.

### Limitations

There's one major limitation to using this data compared to other data sources:

The data scraped here does not specify the venue, and can skew numbers. For example, the Cincinnati Reds were the home team in the inagural [MLB Speedway Classic](https://en.wikipedia.org/wiki/MLB_Speedway_Classic), which drew a crowd of 91,000. Of course, the Reds can't typically draw a crowd like this, so it would produce an outlier in the results. Luckily, occurences like this are rare and should generally not impact the data too much.

### Baseball Reference

Baseball Reference is a great place to view data for baseball, however it can be a little challenging downloading data from them. First, they don't have any public datasets or an API available, so data needs to be scraped from their site. Second, scraping data from their site can be tedious because they rate limit requests to about 20 requests per minute.

### Loading the Data

I've included the full raw data in this repo (`raw_output.csv`) so you don't need to wait for the scraping to run, but if you want to download the data yourself you can run the cell below to fetch all of the data. To minimize waiting time, you can also adjust the `start_year` and `end_year`, to focus only on the years you'd like to view. Running the scraping script for all years (2000-2025) takes about 75 minutes.

I've abstracted away the scraping, but to summarize: this script iterates over a list of teams and years, and collects each team's schedule for that year, writing the data to `raw_output.csv`.

In [ ]:
from data_helper import DataHelper
DataHelper().collect_data()
# DataHelper(start_year=2025, end_year=2025).collect_data()

Using start_year=2025
Using end_year=2025
Created DataHelper!
Finished Diamondbacks
Finished Athletics
Finished Braves
Finished Orioles
Finished Red Sox
Finished Cubs
Finished White Sox
Finished Reds
Finished Guardians
Finished Rockies
Finished Tigers
Finished Astros
Finished Royals
Finished Angels
Finished Dodgers
Finished Marlins
Finished Brewers
Finished Twins
Finished Mets
Finished Yankees
Finished Phillies
Finished Pirates
Finished Padres
Finished Giants
Finished Mariners
Finished Cardinals
Finished Rays
Finished Rangers
Finished Blue Jays
Finished Nationals
Finished Collecting All Data!


### Load the Data

In [71]:
import pandas as pd

df = pd.read_csv('raw_output.csv')
# df.head()

### Cleaning the Data

First, there are several unneeded columns that can be removed:

In [72]:
df = df.drop(columns=['Gm#', 'W/L', 'W-L', 'R', 'RA', 'Inn', 'Win', 'Loss', 'Save', 'Time', 'Streak'])
# df.head()

Next, there are two specific row-by-row cases that need to be accounted for:

 First, by default Baseball Reference repeats its column headers on tables at each new month. This makes it easier to view the data in a web browser, but it doesn't play nice with scraping the data. Those rows each need to be filtered out.
 
 Second is strange issues coming from teams with cancelled/relocated seasons. For example, the 2025 Athletics played [this schedule](https://www.baseball-reference.com/teams/ATH/2025-schedule-scores.shtml), but were originally scheduled to play [this schedule](https://www.baseball-reference.com/teams/OAK/2025-schedule-scores.shtml) before they relocated from Oakland to Sacramento [on their way to Las Vegas](https://en.wikipedia.org/wiki/Oakland_Athletics_relocation_to_Las_Vegas).
 
 Luckily, we can just filter out both of these types of rows by only selecting for rows where `Unnamed: 2` is "boxscore", representing a game which was played and completed.

In [73]:
df = df[df['Unnamed: 2'] == 'boxscore'].drop(columns=['Unnamed: 2'])
# df.head()

Next, there are a lot of duplicate rows in the data set. Because of the way the data was pulled, an entry is made for both teams for each game. For example:

| Date | Tm | Unnamed: 4 | Opp | ... |
| ---- | -- | ---------- | --- | --- |
| Tuesday, Apr 4 | ARI | NaN | PHI | ... |
| Tuesday, Apr 4 | PHI | @ | ARI | ... |

For the Philidelphia Phillies @ Arizona Diamondbacks game on April 4, 2000.

Fixing this requires only looking at rows where `Unnamed: 4` is `NaN`, basically just selecting for home games, based on `Tm`:

In [74]:
df = df[df['Unnamed: 4'].isna()].drop(columns=['Unnamed: 4'])
# df.head()

At this point, the average number of games each team plays should be approximately 81. Sometimes this number will vary, based on cancelled games or relocated games, but in a typical season this will be exactly 81, like in 2025:

In [75]:
print(f'Home games per team: {len(df[df['year'] == 2025]) / 30}')

Home games per team: 81.0


Next, the `Date` field should be stored as a pandas datetime field for graphing purposes later:

In [76]:
# strip any double header information from the raw date string (example: Thursday, Apr 4 (1))
df['Date'] = df['Date'].str.replace(r'\s*\(\d+\)$', '', regex=True)
df['date'] = pd.to_datetime(df['Date'] + ' ' + df['year'].astype(str), format='%A, %b %d %Y')
df = df.drop(columns=['Date'])
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,year,date
0,ARI,PHI,2,0.5,N,44298,.97,NaN,2000,2000-04-04
1,ARI,PHI,1,up 0.5,N,29291,.97,NaN,2000,2000-04-05
2,ARI,PHI,1,up 1.0,N,28774,1.02,NaN,2000,2000-04-06
3,ARI,PIT,1,Tied,N,32536,1.07,NaN,2000,2000-04-07
4,ARI,PIT,1,up 1.0,D,33298,1.02,NaN,2000,2000-04-08


The GB column can also have some mixed data types since the raw data has values like "up 0.5" and "Tied". Also, since these will be graphed, let's convert it into a more graph friendly format; "up 0.5" becomes `0.5`, "Tied" becomes `0`, and 2.5 (two and a half back) becomes `-2.5`.

In [77]:
import re

def parse_gb(row):
    row = str(row).strip()
    # tied with lead = 0
    if row.lower() == 'tied':
        return 0.0
    # some GB columns come through as up10.5 instead of up 10.5, regex the games out
    match = re.match(r'up\s*([\d.]+)', row, re.IGNORECASE)
    if match:
        return float(match.group(1))
    
    return -float(row)

df['GB'] = df['GB'].apply(parse_gb)
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,year,date
0,ARI,PHI,2,-0.5,N,44298,.97,NaN,2000,2000-04-04
1,ARI,PHI,1,0.5,N,29291,.97,NaN,2000,2000-04-05
2,ARI,PHI,1,1.0,N,28774,1.02,NaN,2000,2000-04-06
3,ARI,PIT,1,0.0,N,32536,1.07,NaN,2000,2000-04-07
4,ARI,PIT,1,1.0,D,33298,1.02,NaN,2000,2000-04-08


There's one last scenario to account for: games with no attendance. This can happen for several reasons, but most notably occured during the COVID-19 pandemic. Filtering out rows with no attendance will solve that:

In [78]:
df = df[df['Attendance'].notna()]
# df.head()

Renaming the columns to something more readable and correcting the data types will also be helpful:

In [79]:
df = df.rename(columns={
    'Tm': 'home_team',
    'Opp': 'away_team',
    'Rank': 'rank',
    'GB': 'standings_gap',
    'D/N': 'day_night',
    'Attendance': 'attendance',
    'Orig. Scheduled': 'orig_scheduled'
})

df = df.astype({
    'rank': 'int8',
    'attendance': 'int32',
    'cLI': 'float32'
})

# df.head()

And lastly, we'll need a game number for each team, denoting which home game it is for that season. Since sometimes teams go weeks at a time without playing home games, an x-axis represented by a date can be misleading, since some months will have more games.

In [80]:
df = df.sort_values('date')
df['game_number'] = df.groupby(['home_team', 'year']).cumcount() + 1

Lastly, let's save the processed dataframe to easily refer to later, if we don't want to pull and clean all the data again.

In [81]:
df.to_csv('processed_data.csv')

## Graphing with Plotly and Plotly Dash

For this notebook, I chose to go with [Plotly](https://plotly.com/) and their dashboard/app tool, [Plotly Dash](https://plotly.com/dash/). Plotly is a very powerful charts library - created by a company of the same name - that's built on Javascript to create highly interactive charts and displays. It supports a wide variety of charts and graphs out of the box, with high levels of customization. Plotly Dash is built on top of Plotly, combining a small local webserver (Flask), the Javascript implementation of Plotly, and React.js. This allows you to map your python elements directly to HTML, creating a robust web app without needs to do any HTML or Javascript yourself.

Here's a quick exampl of a histogram using randomly generated data:

In [1]:
import numpy as np
import plotly.express as px

data = np.random.normal(loc=50, scale=10, size=1000)
fig = px.histogram(data, nbins=30, labels={'value': 'Random Value'}, title='Quick Plotly Demo: Histogram of Random Data')
fig.update_layout(showlegend=False, title_x=0.5)
fig

as you can see, you can hover over the elements and have an interactive chart, no extra work required. 

If you want to incorporate more into the display, you can add python blocks that correspond to HTML elements:

In [6]:

from dash import Dash, html, dcc
import plotly.express as px
import numpy as np

# create a new Flask application
app = Dash()

# randomly generate and plot some data
data = np.random.normal(loc=50, scale=10, size=1000)
fig = px.histogram(data, nbins=30, labels={'value': 'Random Value'})
fig.update_layout(showlegend=False, title_x=0.5)

# create an HTML <div> that contains the title in an <h1>
app.layout = html.Div([
    html.H1('Chart Title'),
    html.P('This is a paragraph about the interactive histogram below.'),
    dcc.Graph(figure=fig),
], style={'background-color': 'white', 'text-align': 'center', 'font-family': 'serif', 'padding': '20px'})

# run the Flask application
app.run()

As you can see, the `html.Div({})` object creates a `<div>` within the Flask application and even let's us use CSS style attributes to customize it.

### Installation, Source, and Licensing

Installing both Plotly and Dash is very easy, simply run:

```
pip install plotly
pip install dash
```

The projects are both open source and have MIT (Open Source) licensing. It is worth noting that the company Plotly is a privately held, VC backed company and they do offer cloud and enterprise products that are related to Plotly.

## Plotting MLB Attendance Data

Now let's begin plotting the MLB Attendance data. I think there are four charts that can both highlight the power of Plotly/Dash and give us more insight into MLB attendance. First, let's load the data (if starting from here) and all the Plotly/Dash objects needed, along with some helper data I wrote.

In [16]:
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px
import pandas as pd
from team_helper import get_team_colors, TEAM_NAMES

# load the data if starting from here
try:
    df == None
except NameError:
    df = pd.read_csv('processed_data.csv')

# df.head()

I'm also going to set some hard-coded values to just demonstrate each graph first. Later, these values will be dynamic in the interactive dashboard:

In [ ]:
# which team's data to display, the Detroit Tigers by default
team = 'DET'
# 2025 season data
year = int(df['year'].max())

# filtered down data frame focusing on the team & year
team_df = df[(df['home_team'] == team) & (df['year'] == year)]

### Line Chart

A simple line chart is a great place to start both when working with a new charts library and a new data set. Earlier, I mentioned each team plays about 81 home games and that I numbered these using `game_number` to make sure there was no strange distribution of games between months.

To create a line chart using plotly, you can use `px.line()` and supply it the dataframe, x, and y values. I've also set the line to adjust its color to be the same as the team's primary color, and set a few labels as well. For more information on what you can do with line charts, refer to the [line chart documentation](https://plotly.com/python/line-charts/).

In [18]:
line_graph = px.line(team_df, x='game_number', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
              labels={'game_number': 'Home Game Number', 'attendance': 'Attendance', 'home_team': 'Team'}, title=f'{team} Attendance by Home Game Number')
# hide legend, center title
line_graph.update_layout(showlegend=False, title_x=0.5)
line_graph.show()

I think a line chart is a great place to start exploring this data because it can raise a lot of great questions about the season at a glance. For example, we see quite a few peaks and valleys throughout the year, what causes those? Are the valleys weekday games and the peaks weekend games? Or possibly are the peaks when highly popular teams and/or players are visiting?

One interesting thing to note in this data straight away is that the minimum attendance value happened early in the season, occuring during the fourth, fifth, and sixth home games. While we could explore these data points using the other information at our disposal, we wouldn't find the true reason anywhere in our data: the temperature at game time those days, April 7th, 8th, and 9th, was 38, 34, and 42 degrees respectively.

### Bar Chart

One thing I wanted to explore was if a popular away team drove attendance up for some teams, and I think a bar chart is a great way to do that. Since teams play each other multiple times each year, we can think of `away_team` as a category to average values across.

To create a bar chart using Plotly, you can use `px.bar()` and provide it the data frame, the x values (the numerical values) and y values (the categorical values). In my case, I specifically wanted a horizontal bar chart, so I set `orientation='h'`. Like the line chart above, I set each bar's color value using a map of the team colors, but set them based on the `away_team` value this time. One small thing I had to do was also use `.update_yaxes()` to force all the labels to appear, since by default Plotly was removing some of them. For more information on the Plotly bar charts, refer to [the documentation](https://plotly.com/python/bar-charts/).

In [19]:
# get the average attendance by away team and sort descending 
dff_grouped = team_df.groupby('away_team').agg(avg_attend=('attendance', 'mean')).reset_index().sort_values('avg_attend', ascending=False)

#plot
bar_chart = px.bar(dff_grouped, y='away_team', x='avg_attend', orientation='h',
                   color='away_team', color_discrete_map=get_team_colors(dff_grouped['away_team']),
                   labels={'away_team': 'Opponent', 'avg_attend': 'Average Attendance'},
                   title='Average Attendance by Opponent')

# force all team labels to appear
bar_chart.update_yaxes(tickmode='linear', dtick=1)
# remove legend and center title
bar_chart.update_layout(showlegend=False, title_x=0.5)
bar_chart.show()

Exploring this bar chart is very interesting! I chose a bar chart for this question because it would make direct comparison between the away teams very easy, and we're able to clearly see the differences between the teams the Detroit Tigers hosted.

My original question was if popular teams drive attendance, and this paints an interesting picture in that regard. In Detroit, it seems that the popular teams didn't drive attendance in 2025 (though the NYY games were those cold April games mentioned above). Instead, attendance was seemlingly driven by regional opponents such as Chicago, Toronto, Cincinnati and Cleveland, though some popular team like Atlanta were in the mix as well.

### Scatter Plot

The scatter plot is a very strong tool for evaluating several different points of data in this set. Since I wanted to explore the relationships between other variables and the attendance of games, the scatter plot was a natural fit for this data set. In this case, I want to evaluate the relationship between the championship leverage index (how important the game is to winning the championship) and attendance.

To create a scatter plot in Plotly, you can run `px.scatter()` and pass the data frame, the x values, and y values. Like the other charts so far, I'm also specifying certain colors for the dots on the plot. As you can see below I'm also annotating the scatter plot with the correlation coefficient (from pandas) using `.add_annotation()`. For more information, refer to the [scatter plot documentation](https://plotly.com/python/line-and-scatter/).

In [26]:
# plot
scatter_plot = px.scatter(team_df, x='cLI', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                          labels={'cLI': 'Championship Leverage Index (Game Importance)', 'attendance': 'Attendance'},
                          title='Attendance vs. Game Importance', trendline='ols')

# add correlation
r_value = team_df['cLI'].corr(team_df['attendance'])
scatter_plot.add_annotation(text=f'r = {r_value:.3f}', xref='paper', yref='paper', x=0.98, y=0.98, showarrow=False, font=dict(size=14))

# remove legend and center title
scatter_plot.update_layout(showlegend=False, title_x=0.5)
scatter_plot.show()

Again, since I wanted to directly explore the relationship between attendance and game importance a scatter plot was a natural fit. However, it seems like this graph shows only a loose correlation between the game importance and attendance for the 2025 Tigers. Part of this could be that the Tigers held a large lead through most of the year, and played relatively few highly important games.

### Jittered Dot and Box Plot

One of the questions I asked early on when evaluating the line chart was if there was lower attendance on weeknights than on weekend games. I also was wondering if day or night games tended to have more attendance. There tends to be a day of the week pattern of when day or night games are played, so I thought it would be best to highlight the different game times as well. To answer this I wanted to use a jittered dot plot with a box plot overlay, where the dots were differently colored based on day or night games..

You can create the jittered dot plot using `px.strip()`, and again pass the data frame, x and y values. This time I assigned dot colors based on day or night games. More information on jittered dot plots can be found in the [documentation](https://plotly.com/python/strip-charts/).

To create the box plots, run `px.box()` and pass the data frame, x and y values. I wanted to directly control the opacity and line width of my boxplot to prevent it from overpowering the dot plot, so I manually set those using `line=dict(color='black', width=1)` and `opacity=0.4`.

Then to overlay the box plot, I used the following code to set the two different plots to overlay and add each box plot on top.
```
jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
for trace in box_overlay.data:
    jittered_dot_plot.add_trace(trace)
```

In [27]:
# assign a day of the week to each game and establish the order of the week
team_df['day_of_week'] = pd.to_datetime(team_df['date']).dt.day_name()
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# plot the jittered dot
jittered_dot_plot = px.strip(
    team_df, x='day_of_week', y='attendance', color='day_night',
    category_orders={'day_of_week': day_order},
    color_discrete_map={'D': '#F5A623', 'N': '#1B1F3B'},
    labels={'day_of_week': 'Day of Week', 'attendance': 'Attendance', 'day_night': 'Day/Night'},
    title='Attendance by Day of Week and Game Time'
)

# create a box plot too
box_overlay = px.box(team_df, x='day_of_week', y='attendance', category_orders={'day_of_week': day_order})
box_overlay.update_traces(fillcolor='rgba(255,255,255,0)', line=dict(color='black', width=1), opacity=0.4, showlegend=False)

# overlay the box plot on the jittered dot
jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
for trace in box_overlay.data:
    jittered_dot_plot.add_trace(trace)

jittered_dot_plot.show()

My goal was to evaluate if weeknight or weekend games had higher or lower attendance and this graph shows me exactly what I was looking for! The medians values for each weekend day were each higher than the maximums for each week day (except Monday, which gets a boon from Labor Day). The usage of the jittered dot / box plots were a great way to see the distribution of attendance based on the day of the week.

## Plotly Dash Dashboard

Now it's time to put all of these charts together and make them a bit more dynamic. Right now to view data for a different team from a different year, you would need to manually set the team code and year and re-run all of the charts again. 

However, we can use the HTML building blocks above to not only render graphs together, but control inputs using HTML input elements to allow for different teams or years to be selected instead. When looking at the code below, imagine the `app.layout` variable to look like the following HTML:
```
<div>
    <h1>Attendance Data by Team and Year</h1>
    <div>
        <div>
            <select>
            <!-- Team Options -->
            ...
            </select>
        </div>
        <div>
            <select>
            <!-- Year Options -->
            ...
            </select>
        </div>
    </div>
    <div>
        <div>
            <graph>
                <!-- Line Chart -->
            </graph>
            <graph>
                <!-- Bar Chart -->
            </graph>
        </div>
        <div>
            <graph>
                <!-- Scatter Plot -->
            </graph>
            <graph>
                <!-- Jittered Dot / Box Plot -->
            </graph>
        </div>
    </div>
</div>
```

In [ ]:
app = Dash()

app.layout = html.Div([
    html.H1(children='Attendance Data by Team and Year', style={'textAlign':'center'}),
    html.Div([
        html.Div(
            dcc.Dropdown(
                # get real team names from my helper file
                # HTML for this is like <option value="DET">Detroit Tigers</option>
                [{'label': TEAM_NAMES.get(code, code), 'value': code} for code in sorted(df['home_team'].unique(), key=lambda c: TEAM_NAMES.get(c, c))],
                'DET', id='team-dropdown'
            ),
            style={'width': '70%', 'padding': '10px'}
        ),
        html.Div(
            dcc.Dropdown(sorted(df['year'].unique()), df['year'].max(), id='year-dropdown'),
            style={'width': '30%', 'padding': '10px'}
        ),
    ], style={'display': 'flex', 'flexDirection': 'row'}),

    html.Div([
        html.Div([
            dcc.Graph(id='attendance-by-team-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-opponent-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),

        html.Div([
            dcc.Graph(id='attendance-by-game-importance-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-day-or-night-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),
    ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
], style={'backgroundColor': '#ffffff', 'color': '#111', 'minHeight': '100vh', 'padding': '20px', 'font-family': 'sans-serif'})

But this code won't make the dashboard response quite yet. Remember that this was only the HTML; what makes it dynamic and responsive is Javascript to update the values based on our inputs.

To accomplish this, we use a method decorator, `@callback`, to specify which elements in our HTML above are inputs and which ones are outputs. Then the method it's decorating takes those same inputs as parameters, and we add the chart logic we defined above to have it generate charts each time our inputs are updated.

One note, I've also added `.update_xaxes()` or `.update_yaxes()` to the charts as needed to control the global mins/maxs of each chart now. Since we can now change teams quickly, teams should be held to the same scope if they're in the same year.

Lastly, I chose to use `app.run(jupyter_mode='tab')` here instead to open the output in a new browser tab. Inside a notebook, it can produce a slightly cramped output versus a web browser.

In [32]:
@callback(
    Output('attendance-by-team-graph', 'figure'),
    Output('attendance-by-opponent-graph', 'figure'),
    Output('attendance-by-game-importance-graph', 'figure'),
    Output('attendance-by-day-or-night-graph', 'figure'),
    Input('team-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_graph(team, year):
    dff = df[(df['home_team'] == team) & (df['year'] == year)]
    year_df = df[df['year'] == year]
    attend_min, attend_max = year_df['attendance'].min(), year_df['attendance'].max()

    # --- line chart ---
    line_graph = px.line(dff, x='game_number', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                  labels={'game_number': 'Home Game Number', 'attendance': 'Attendance', 'home_team': 'Team'}, title='Attendance Home Game Number')
    line_graph.update_yaxes(range=[attend_min, attend_max])
    line_graph.update_layout(showlegend=False, title_x=0.5)

    # --- bar chart ---
    dff_grouped = dff.groupby('away_team').agg(avg_attend=('attendance', 'mean')).reset_index()
    dff_grouped = dff_grouped.sort_values('avg_attend', ascending=False)
    bar_chart = px.bar(dff_grouped, y='away_team', x='avg_attend', orientation='h',
                       color='away_team', color_discrete_map=get_team_colors(dff_grouped['away_team']),
                       labels={'away_team': 'Opponent', 'avg_attend': 'Average Attendance'},
                       title='Average Attendance by Opponent')
    max_avg_attendance = year_df.groupby(['home_team', 'away_team'])['attendance'].mean().max()
    bar_chart.update_xaxes(range=[0, max_avg_attendance])
    bar_chart.update_yaxes(tickmode='linear', dtick=1)
    bar_chart.update_layout(showlegend=False, title_x=0.5)

    # --- scatter plot ---
    scatter_plot = px.scatter(dff, x='cLI', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                              labels={'cLI': 'Championship Leverage Index (Game Importance)', 'attendance': 'Attendance'},
                              title='Attendance vs. Game Importance')
    scatter_plot.update_yaxes(range=[attend_min, attend_max])
    scatter_plot.update_layout(showlegend=False, title_x=0.5)

    # --- jittered dot plot + box overlay ---
    dff['day_of_week'] = pd.to_datetime(dff['date']).dt.day_name()
    day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    jittered_dot_plot = px.strip(
        dff, x='day_of_week', y='attendance', color='day_night',
        category_orders={'day_of_week': day_order},
        color_discrete_map={'D': '#F5A623', 'N': '#1B1F3B'},
        labels={'day_of_week': 'Day of Week', 'attendance': 'Attendance', 'day_night': 'Day/Night'},
        title='Attendance by Day of Week and Game Time'
    )
    box_overlay = px.box(dff, x='day_of_week', y='attendance', category_orders={'day_of_week': day_order})
    box_overlay.update_traces(fillcolor='rgba(255,255,255,0)', line=dict(color='black', width=1), opacity=0.4, showlegend=False)
    jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
    for trace in box_overlay.data:
        jittered_dot_plot.add_trace(trace)

    return line_graph, bar_chart, scatter_plot, jittered_dot_plot

app.run(jupyter_mode='tab')

Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>